# Catégorisation RSE des Libellés APE — EcoVadis
**Text Mining en Python** — Dictionnaire enrichi sur échantillon réel (722 libellés)

Il réalise une classification automatique des codes NAF/APE selon les composantes RSE EcoVadis :
- **ENVIRONNEMENT**
- **SOCIAL & DROITS HUMAINS**
- **ÉTHIQUE**
- **ACHATS RESPONSABLES**

## 1. Imports et installation des dépendances

In [1]:
import pandas as pd
import re
import unicodedata
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

## 2. Import des données

Le fichier Excel doit contenir les colonnes suivantes :
- `SIREN`
- `Dénomination sociale de l'entreprise`
- `Nom commercial`
- `Date societe a mission`
- `Date de création`
- `Code NAF - APE`
- `Libelle APE`
- `Date de vote mission`

In [2]:
# Chemin vers le fichier source — à adapter selon votre environnement
CHEMIN_FICHIER = "export_univ_creteil.xlsx"

df = pd.read_excel(CHEMIN_FICHIER)

df.head(3)

,SIREN,Dénomination sociale de l'entreprise,Nom commercial de l'entreprise,Date societe a mission,Date de création,Code NAF - APE,Libelle APE,Date de vote de la mission par les actionnaires
0,948955356,BIASSA,BIASSA,2023-02-15,19/1/2023,3811Z,Collecte des déchets non dangereux,2023-01-19
1,383699048,RAMSAY SANTE,Ramsay Santé,2023-02-13,29/11/1991,6430Z,Fonds de placement et entités financières simi...,2022-12-08
2,918950049,DIPTICK,Diptick,2023-02-15,6/9/2022,5829C,Édition de logiciels applicatifs,2023-02-08


## 3. Dictionnaire de mots-clés RSE enrichi

Structure : `{ composante: { sous_theme: [mots_cles] } }`

Les mots-clés sont écrits **sans accents** car la normalisation les supprime avant la comparaison.
Les points (`.`) dans certains mots-clés remplacent les apostrophes (ex: `d.eau` → `d eau`).

In [3]:
#Keywords_rse générés par une I.A. 

keywords_rse = {

    # ============================================================
    "ENVIRONNEMENT": {

        "Consommation d'energie et emissions de GES": [
            "electricite", "commerce d.electricite", "commerce de combustibles",
            "gaz", "combustibles gazeux", "petrole", "carburant",
            "moteur", "generateur", "transformateur", "thermique",
            "eolien", "solaire", "photovoltaique", "nucleaire",
            "energie", "production d.electricite",
            "transport routier", "transport fluvial", "transport de voyageurs",
            "taxis", "fret", "logistique", "entreposage", "stockage",
            "chauffage", "climatisation", "equipements thermiques"
        ],

        "Eau": [
            "captage", "traitement et distribution d.eau", "industrie des eaux",
            "eaux de table", "eau", "hydraulique", "assainissement",
            "irrigation", "piscicult", "aquaculture", "marin", "fluvial",
            "meunerie", "brasserie", "boisson", "travaux de distribution d.eau",
            "construction de reseaux pour fluides"
        ],

        "Biodiversite": [
            "agricult", "culture de la vigne", "vigne", "viticulture",
            "elevage", "sylviculture", "activites forestieres", "foret", "bois",
            "peche", "chasse", "faune", "flore", "espaces verts",
            "amenagement paysager", "paysager", "jardinage", "parc",
            "extraction miniere", "carriere"
        ],

        "Pollution locale et accidentelle": [
            "chimique", "chimie", "petrochimie", "raffinage",
            "extraction", "minier", "minerai", "metallurgie", "fonderie",
            "siderurgie", "traitement de surface", "nettoyage courant",
            "nettoyage industriel", "blanchisserie", "teinturerie",
            "station d.epuration", "hydrocarbure", "peinture et vitrerie",
            "peinture", "vitrerie"
        ],

        "Materiaux produits chimiques et dechets": [
            "dechet", "collecte des dechets", "traitement et elimination",
            "recuperation de dechets", "recyclage", "valorisation",
            "plastique", "matieres plastiques", "elements en matieres plastiques",
            "emballage", "cartonnage", "papier", "carton", "verre",
            "fabrication de cartonnages", "imprimerie",
            "transformation du the", "transformation et conservation",
            "travail des grains"
        ],

        "Utilisation des produits": [
            "electromenager", "automobile", "vehicule", "voitures",
            "appareils d.eclairage", "equipement electrique",
            "electronique", "ordinateurs", "equipements peripheriques",
            "materiels optique", "photographique",
            "instruments de musique", "jeux et jouets",
            "articles de sport", "articles de bijouterie",
            "bijouterie fantaisie", "joaillerie", "bijouterie",
            "materiel medico-chirurgical", "dispositif medical",
            "appareils d.eclairage electrique",
            "fabrication de machines", "autres machines specialisees",
            "charpentes", "menuiserie", "articles metalliques",
            "huiles essentielles"
        ],

        "Fin de vie des produits": [
            "recuperation de dechets tries", "traitement et elimination des dechets",
            "dechets non dangereux", "collecte des dechets",
            "reparation de machines", "reparation d.equipements",
            "reparation de chaussures", "reparation d.autres biens",
            "reparation d.equipements de communication"
        ],

        "Sante et securite des clients": [
            "agroalimentaire", "aliments", "alimentaire",
            "restauration traditionnelle", "restauration de type rapide",
            "restauration collective", "cafeteria", "libre-service",
            "traiteur", "pharmacie", "preparations pharmaceutiques",
            "medicament", "dispositif medical", "materie medico",
            "produits pharmaceutiques", "cosmetique", "parfum",
            "produits de beaute", "parfumerie", "toilette",
            "aliments homogeneises", "dietetiques",
            "aliments pour animaux", "alimentation generale",
            "commerce d.alimentation",
            "transformation et conservation de fruits",
            "fabrication de plats prepares", "biscuits", "biscottes",
            "patisseries", "autres produits alimentaires",
            "the", "cafe", "cacao", "epices",
            "boissons fermentees", "biere", "vins effervescents",
            "articles medicaux", "orthopediques", "optique"
        ],

        "Consommation durable": [
            "vente a distance", "catalogue general", "catalogue specialise",
            "commerce de detail", "commerce de gros",
            "grande distribution", "vente a domicile",
            "biens d.occasion", "autres commerces de detail",
            "commerce d.alimentation generale"
        ]
    },

    # ============================================================
    "SOCIAL & DROITS HUMAINS": {

        "Sante et securite des employes": [
            "construction", "travaux", "batiment", "genie civil", "chantier",
            "construction d.autres batiments", "construction de reseaux",
            "construction d.autres ouvrages",
            "extraction", "minier", "industrie lourde",
            "nettoyage courant des batiments", "nettoyage industriel",
            "securite privee", "systemes de securite", "gardiennage",
            "ambulance", "hospitalier", "sante humaine",
            "demenagement", "menuiserie bois", "menuiserie metallique",
            "serrurerie", "travaux d.installation",
            "economiste de la construction"
        ],

        "Conditions de travail": [
            "restauration", "hotel", "hebergement", "hotellerie",
            "hebergement similaire", "hebergement touristique",
            "hebergement de courte duree",
            "agriculture", "elevage",
            "interim", "agences de travail temporaire",
            "placement de main-d.oeuvre",
            "nettoyage", "entretien", "services a la personne",
            "aide a domicile", "garde d.enfant", "accueil de jeunes enfants",
            "livraison", "poste", "service universel",
            "centres d.appels", "services administratifs",
            "blanchisserie", "teinturerie"
        ],

        "Dialogue social": [
            "syndicat", "dialogue social", "relations sociales",
            "mediation", "arbitrage", "conseil social"
        ],

        "Gestion des carrieres et formation": [
            "enseignement", "formation continue", "formation d.adultes",
            "ecole", "universite", "enseignement superieur",
            "academie", "centre de formation", "apprentissage",
            "education", "pedagogie", "soutien scolaire", "e-learning",
            "soutien a l.enseignement", "autres enseignements"
        ],

        "Travail des enfants et travail force": [
            "textile", "confection", "habillement", "vetement",
            "vetements de dessus", "vetements de travail",
            "chaussure", "maroquinerie", "cuir",
            "tissage", "filature", "fibres textiles",
            "sous-traitance internationale",
            "intermediaires du commerce en text"
        ],

        "Diversite discrimination et harcelement": [
            "ressources humaines", "recrutement",
            "cabinet de recrutement", "conseil rh",
            "travail temporaire", "interim",
            "placement de main-d.oeuvre",
            "agences de placement"
        ],

        "Droits humains externes": [
            "aide humanitaire", "cooperation internationale",
            "ong", "fondation", "developpement international",
            "action sociale", "action sociale sans hebergement",
            "hebergement social", "accompagnement",
            "adultes handicapes", "personnes agees",
            "accueil ou accompagnement", "accueil de jeunes"
        ]
    },

    # ============================================================
    "ETHIQUE": {

        "Corruption": [
            "finance", "banque", "intermediations monetaires",
            "assurance", "assurance vie", "autres assurances",
            "agents et courtiers d.assurances",
            "fonds", "fonds de placement", "gestion de fonds",
            "investissement", "placement", "bourse", "courtage",
            "audit", "activites comptables", "comptabilite",
            "commissaire aux comptes", "fiscalite",
            "juridique", "activites juridiques", "avocat",
            "notaire", "huissier", "fiduciaire",
            "entites financieres", "autres intermediations",
            "distribution de credit", "autre distribution de credit",
            "acti. auxi. de services fin.",
            "activites auxiliaires de services financiers",
            "activites des societes holding",
            "sieges sociaux", "sieges sociaux non alimentaires",
            "supports juridiques de gestion de patrimoine",
            "gestion d.immeubles"
        ],

        "Pratiques anticoncurrentielles": [
            "conseil pour les affaires", "conseils de gestion",
            "conseil en relations publiques",
            "publicite", "marketing", "communication",
            "medias", "relations publiques",
            "agences de publicite", "presse",
            "edition de livres", "edition de revues",
            "edition de periodiques", "autre creation artistique",
            "enregistrement sonore", "edition musicale",
            "etudes de marche", "sondages",
            "organisation de foires", "salons professionnels", "congres",
            "voyagistes", "agences de voyage",
            "activites recreatives", "loisirs",
            "clubs de sports", "activites liees au sport",
            "soutien au spectacle vivant",
            "production de films"
        ],

        "Gestion responsable de l.information": [
            "logiciel", "logiciels applicatifs", "logiciels systeme",
            "editeur de logiciels",
            "informatique", "numerique", "digital",
            "programmation informatique",
            "developpement informatique",
            "systemes et logiciels informatiques",
            "conseil en systemes",
            "systeme d.information", "cybersecurite",
            "donnees", "cloud", "intelligence artificielle",
            "telecommunication", "telecommunications par satellite",
            "autres activites de telecommunication",
            "internet", "portails internet",
            "traitement de donnees", "hebergement",
            "gestion d.installations informatiques",
            "activites connexes"
        ]
    },

    # ============================================================
    "ACHATS RESPONSABLES": {

        "Pratiques environnementales des fournisseurs": [
            "ingenierie", "bureau d.etudes", "etudes techniques",
            "architecture", "activites d.architecture",
            "urbanisme", "achat", "approvisionnement",
            "supply chain", "chaine d.approvisionnement",
            "commerce de gros", "grossiste", "negoce",
            "intermediaires du commerce",
            "autres intermediaires du commerce",
            "analyses essais inspections", "analyses, essais",
            "recherche-developpement", "recherche developpement",
            "activites specialisees scientifiques",
            "activites specialisees, scientifiques",
            "biotechnologie",
            "autres activites de soutien aux entreprises",
            "activites de soutien",
            "services de reservation",
            "location et location-bail",
            "location de logements", "location de terrains",
            "immobilier", "agences immobilieres",
            "promotion immobiliere", "marchands de biens immobiliers",
            "administration d.immeubles",
            "loc.-bail de propr. intel"
        ],

        "Pratiques sociales des fournisseurs": [
            "import", "export", "negoce international",
            "commerce international", "agent commercial",
            "interm. du comm.", "intermediaires du commerce",
            "comm. interent.", "interentreprises"
        ]
    }
}

print(f"Dictionnaire : {len(keywords_rse)} composantes RSE")
for comp, themes in keywords_rse.items():
    print(f"   - {comp} : {len(themes)} sous-thèmes")

Dictionnaire : 4 composantes RSE
   - ENVIRONNEMENT : 9 sous-thèmes
   - SOCIAL & DROITS HUMAINS : 7 sous-thèmes
   - ETHIQUE : 3 sous-thèmes
   - ACHATS RESPONSABLES : 2 sous-thèmes


## 4. Normalisation du texte

Cette fonction de normalisation :

1. Met en minuscules
2. Supprime les accents (décomposition Unicode)
3. Supprime les apostrophes et caractères similaires
4. Supprime les espaces superflus

In [4]:
def normaliser(texte: str) -> str:
    if not isinstance(texte, str):
        return ""
    # Minuscules
    texte = texte.lower()
    # Décomposition Unicode et suppression des diacritiques
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    # Suppression des apostrophes et guillemets
    texte = re.sub(r"[''`\u009c]", "", texte)
    # Suppression des espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte


# Test rapide de la normalisation
exemples = ["Fabrication d'équipements électriques", "Hôtellerie – hébergement", "Activités financières"]
for ex in exemples:
    print(f"  '{ex}' → '{normaliser(ex)}'")

  'Fabrication d'équipements électriques' → 'fabrication dequipements electriques'
  'Hôtellerie – hébergement' → 'hotellerie – hebergement'
  'Activités financières' → 'activites financieres'


## 5. Classifieur RSE

Pour chaque libellé :
1. On normalise le texte
2. On compte le nombre de mots-clés qui correspondent, par composante et sous-thème
3. On retourne la combinaison **(composante, sous-thème)** ayant le **score maximum**
4. En cas d'absence de match → `"Non classifie"`

In [5]:
def categoriser_rse(libelle, keywords_dict: dict) -> dict:
    # Cas libellé vide ou manquant
    if not isinstance(libelle, str) or libelle.strip() == "":
        return {"composante": "Non renseigne", "sous_theme": "Libelle vide ou manquant"}

    libelle_clean = normaliser(libelle)
    scores = {}  # clé = (composante, sous_theme), valeur = nb de matchs

    for composante, themes in keywords_dict.items():
        for sous_theme, mots_cles in themes.items():
            # On normalise aussi les mots-clés
            mots_clean = [normaliser(mk) for mk in mots_cles]
            # On construit un pattern regex « mot | mot | … »
            # re.escape pour éviter les problèmes avec les points et tirets
            pattern = "|".join(re.escape(mk) for mk in mots_clean if mk)
            nb_matchs = len(re.findall(pattern, libelle_clean))
            if nb_matchs > 0:
                scores[(composante, sous_theme)] = nb_matchs

    # Aucun mot-clé détecté
    if not scores:
        return {"composante": "Non classifie", "sous_theme": "Aucun mot-cle correspondant"}

    # Sélection de la combinaison avec le score le plus élevé
    best_key = max(scores, key=scores.get)
    return {"composante": best_key[0], "sous_theme": best_key[1]}


# --- Tests unitaires rapides ---
tests = [
    "Construction de bâtiments résidentiels",
    "Activités de conseil pour les affaires et la gestion",
    "Collecte et traitement des déchets",
    "",
    None
]
for t in tests:
    res = categoriser_rse(t, keywords_rse)
    print(f"  '{t}' → {res['composante']} / {res['sous_theme']}")

  'Construction de bâtiments résidentiels' → SOCIAL & DROITS HUMAINS / Sante et securite des employes
  'Activités de conseil pour les affaires et la gestion' → ETHIQUE / Pratiques anticoncurrentielles
  'Collecte et traitement des déchets' → ENVIRONNEMENT / Materiaux produits chimiques et dechets
  '' → Non renseigne / Libelle vide ou manquant
  'None' → Non renseigne / Libelle vide ou manquant


## 6. Application à l'ensemble du DataFrame

In [6]:
resultats = df["Libelle APE"].apply(
    lambda libelle: categoriser_rse(libelle, keywords_rse)
)

# Décomposition du dictionnaire résultat en deux colonnes
df["Type_de_RSE"] = resultats.apply(lambda x: x["composante"])
df["Sous_theme"]  = resultats.apply(lambda x: x["sous_theme"])

df[["SIREN", "Libelle APE", "Type_de_RSE", "Sous_theme"]].head(10)

,SIREN,Libelle APE,Type_de_RSE,Sous_theme
0,948955356,Collecte des déchets non dangereux,ENVIRONNEMENT,Materiaux produits chimiques et dechets
1,383699048,Fonds de placement et entités financières simi...,ETHIQUE,Corruption
2,918950049,Édition de logiciels applicatifs,ETHIQUE,Gestion responsable de l.information
3,900650136,"Autres acti. auxi. de services fin., hors assu...",ETHIQUE,Corruption
4,948686225,Vente à distance sur catalogue spécialisé,ENVIRONNEMENT,Consommation durable
5,948671276,Activités de soutien à l'enseignement,SOCIAL & DROITS HUMAINS,Gestion des carrieres et formation
6,839203684,Conseil pour les affaires et autres conseils d...,ETHIQUE,Pratiques anticoncurrentielles
7,525286894,Conseil en relations publiques et communication,ETHIQUE,Pratiques anticoncurrentielles
8,948900675,"Ingénierie, études techniques",ACHATS RESPONSABLES,Pratiques environnementales des fournisseurs
9,845291285,Conseil en relations publiques et communication,ETHIQUE,Pratiques anticoncurrentielles


## 7. Diagnostics et statistiques

In [7]:
# --- Répartition par composante RSE ---
print(df["Type_de_RSE"].value_counts().to_string())

# --- Taux de classification ---
n_total     = len(df)
n_classifie = df[~df["Type_de_RSE"].isin(["Non classifie", "Non renseigne"])].shape[0]
n_na        = (df["Type_de_RSE"] == "Non renseigne").sum()
n_nc        = (df["Type_de_RSE"] == "Non classifie").sum()

print(f"\n--- Taux de classification ---")
print(f"Total          : {n_total}")
print(f"Classifiés     : {n_classifie} ({100 * n_classifie / n_total:.1f}%)")
print(f"Non renseignés : {n_na} ({100 * n_na / n_total:.1f}%)")
print(f"Non classifiés : {n_nc} ({100 * n_nc / n_total:.1f}%)")

# --- Libellés non classifiés (à enrichir manuellement) ---
print("\n--- Libellés non classifiés (à enrichir manuellement) ---")
non_classifies = (
    df[df["Type_de_RSE"] == "Non classifie"]["Libelle APE"]
    .value_counts()
    .reset_index()
)
non_classifies.columns = ["Libelle APE", "n"]
print(non_classifies.to_string(index=False))

Type_de_RSE
ETHIQUE                    1149
ENVIRONNEMENT               428
ACHATS RESPONSABLES         325
SOCIAL & DROITS HUMAINS     188
Non classifie                93
Non renseigne                38

--- Taux de classification ---
Total          : 2221
Classifiés     : 2090 (94.1%)
Non renseignés : 38 (1.7%)
Non classifiés : 93 (4.2%)

--- Libellés non classifiés (à enrichir manuellement) ---
                                                                                                       Libelle APE  n
                                                                                 Autres services personnels n.c.a. 14
                                                                                  Activités spécialisées de design  8
                                                                     Transports urbains et suburbains de voyageurs  6
                                                                                 Gestion d'installations sportives  4
         

## 8. Export des résultats : format csv

In [8]:
df.to_csv("resultats_categorisation_rse.csv", index=False, sep=";", encoding="utf-8-sig")